# Async Conversion — Modality Timestamp Plot
Visualizes the timestamps of each modality relative to the lidar reference.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from pyarrow import ipc

from py123d.api.utils.arrow_schema import (
    BOX_DETECTIONS_SE3,
    EGO_STATE_SE3,
    LIDAR,
    PINHOLE_CAMERA,
    SYNC,
)

In [ ]:
# Point this to a converted log directory that contains sync.arrow
LOG_DIR = Path("/home/daniel/py123d_workspace/data/logs/nuscenes-mini_train/scene-0553")
assert (LOG_DIR / "sync.arrow").exists(), f"sync.arrow not found in {LOG_DIR}"

lidar_position = "Start"

In [ ]:
# Load tables
sync_table = ipc.open_file(LOG_DIR / "sync.arrow").read_all()


def load_timestamps_us(filename: str, ts_col: str) -> list:
    path = LOG_DIR / filename
    if not path.exists():
        return []
    table = ipc.open_file(path).read_all()
    return table[ts_col].to_pylist()


# Reference: lidar start timestamps
lidar_ts = load_timestamps_us("lidar.arrow", LIDAR.col("start_timestamp_us"))

# Ego
ego_ts = load_timestamps_us("ego_state_se3.arrow", EGO_STATE_SE3.col("timestamp_us"))

# Box detections — no per-row timestamp column, use sync table indices
# We'll read the sync table's timestamp for each box detection row instead
box_ts_col = BOX_DETECTIONS_SE3.prefix()
box_ts = []
if box_ts_col in sync_table.column_names:
    sync_ts = sync_table[SYNC.col("timestamp_us")].to_pylist()
    for i, indices in enumerate(sync_table[box_ts_col].to_pylist()):
        if indices and len(indices) > 0:
            box_ts.append(sync_ts[i])

# Pinhole cameras — collect per-camera timestamps
camera_timestamps = {}
pcam_path = LOG_DIR / "pinhole_camera.arrow"
if pcam_path.exists():
    pcam_table = ipc.open_file(pcam_path).read_all()
    ts_all = pcam_table[PINHOLE_CAMERA.col("timestamp_us")].to_pylist()
    cam_ids = pcam_table[PINHOLE_CAMERA.col("camera_id")].to_pylist()
    for ts, cid in zip(ts_all, cam_ids):
        camera_timestamps.setdefault(cid, []).append(ts)

print(f"Lidar sweeps:  {len(lidar_ts)}")
print(f"Ego states:    {len(ego_ts)}")
print(f"Box det frames:{len(box_ts)}")
print(f"Camera IDs:    {sorted(camera_timestamps.keys())}")
for cid, cts in sorted(camera_timestamps.items()):
    print(f"  cam {cid}: {len(cts)} frames")

In [ ]:
# Build modality list: (label, timestamps)
from py123d.datatypes.sensors.pinhole_camera import PinholeCameraID

ref_ts = min(lidar_ts) if lidar_ts else 0


def to_milli_seconds(ts_list):
    return [(t - ref_ts) / 1e3 for t in ts_list]


modalities = []
modalities.append((f"Lidar {lidar_position}", to_milli_seconds(lidar_ts)))
modalities.append(("Ego Pose", to_milli_seconds(ego_ts)))
if box_ts:
    modalities.append(("Box Det.", to_milli_seconds(box_ts)))
for cid in sorted(camera_timestamps.keys()):
    modalities.append((f"{PinholeCameraID(cid).serialize()}", to_milli_seconds(camera_timestamps[cid])))

# Plot
fig, ax = plt.subplots(figsize=(18, 0.5 * len(modalities) + 1.5))

colors = plt.cm.tab10.colors

for y_idx, (label, ts) in enumerate(modalities):
    color = colors[0] if label == "Lidar" else colors[y_idx % len(colors)]
    marker = "|" if label != "Lidar" else "|"
    ax.scatter(ts, [y_idx] * len(ts), marker=marker, s=30, linewidths=0.8, color=color, label=label)

# Draw light vertical lines at lidar timestamps as reference
lidar_s = to_milli_seconds(lidar_ts)
for lt in lidar_s:
    ax.axvline(lt, color=colors[0], alpha=0.08, linewidth=0.5)

ax.set_yticks(range(len(modalities)))
ax.set_yticklabels([m[0] for m in modalities])
ax.set_xlabel("Time (ms) relative to first lidar sweep")
ax.set_title("Modality Timestamps")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Zoomed view: first 2 seconds
fig, ax = plt.subplots(figsize=(10, 0.25 * len(modalities) + 1.5))

for y_idx, (label, ts) in enumerate(modalities):
    color = colors[0] if label == "Lidar" else colors[y_idx % len(colors)]
    ax.scatter(ts, [y_idx] * len(ts), marker="|", s=180, linewidths=2, color=color)

for lt in lidar_s:
    ax.axvline(lt, color=colors[0], alpha=0.15, linewidth=0.5)


start = 200
duration = 1000
offset = 10
ax.set_xlim(start - offset, start + duration + offset)
ax.set_yticks(range(len(modalities)))
ax.set_yticklabels([m[0] for m in modalities])
ax.set_xlabel("Time (ms) relative to first lidar sweep")
# ax.set_title("Modality Timestamps")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()